# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from tavily import TavilyClient
from pydantic import BaseModel

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import SystemMessage, UserMessage
from lib.tooling import tool


In [3]:
load_dotenv("../.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


In [4]:
class EvaluationReport(BaseModel):
    useful: bool
    description: str


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
@tool
def retrieve_game(query: str) -> str:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    client = chromadb.PersistentClient(path="chromadb")
    emb_fn = embedding_functions.OpenAIEmbeddingFunction(
        api_key=OPENAI_API_KEY,
        model_name="text-embedding-3-small"
    )
    collection = client.get_collection(name="udaplay", embedding_function=emb_fn)
    results = collection.query(query_texts=[query], n_results=3)

    if not results["documents"][0]:
        return "No games found matching the query."

    output = []
    for metadata in results["metadatas"][0]:
        output.append(
            f"- Platform: {metadata.get('Platform')}, "
            f"Name: {metadata.get('Name')}, "
            f"Year: {metadata.get('YearOfRelease')}, "
            f"Description: {metadata.get('Description')}"
        )
    return "\n".join(output)


#### Evaluate Retrieval Tool

In [6]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini")
    messages = [
        SystemMessage(
            content=(
                "Your task is to evaluate if the documents are enough to respond the query. "
                "Give a detailed explanation, so it's possible to take an action to accept it or not."
            )
        ),
        UserMessage(
            content=f"Question: {question}\n\nRetrieved Documents:\n{retrieved_docs}"
        ),
    ]
    response = llm.invoke(messages, response_format=EvaluationReport)
    report = EvaluationReport.model_validate_json(response.content)
    return f"Useful: {report.useful}\nDescription: {report.description}"


#### Game Web Search Tool

In [7]:
@tool
def game_web_search(question: str) -> str:
    """
    Web search: Finds most relevant results from the web
    args:
    - question: a question about game industry.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question, search_depth="basic", max_results=3)
    results = response.get("results", [])
    if not results:
        return "No web results found."
    output = []
    for r in results:
        output.append(
            f"- {r.get('title')}: {r.get('content')} "
            f"(Source: {r.get('url')})"
        )
    return "\n".join(output)


### Agent

In [8]:
MODEL = "gpt-4o-mini"

INSTRUCTIONS = """You are UdaPlay, an expert AI research agent for the video game industry.

When answering a question:
1. Call retrieve_game to search the local game database.
2. Call evaluate_retrieval with the question and the retrieved results to assess their usefulness.
3. If the evaluation says the results are NOT useful, call game_web_search to find up-to-date information.
4. Synthesize information from all sources and give a clear, well-structured answer.
5. Always indicate whether the answer came from the local database, the web, or both."""

agent = Agent(
    model_name=MODEL,
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search]
)


In [9]:
queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)
    run = agent.invoke(query, session_id="udaplay_test")
    final_state = run.get_final_state()
    messages = final_state.get("messages", [])
    if messages:
        print(f"Answer:\n{messages[-1].content}")



Query: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_router
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_router
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Answer:
Pokémon Gold and Silver were released in 1999 for the Game Boy Color. This information comes from the local game database.

Query: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_router
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_router
[StateMachine] Execut

### (Optional) Advanced

### Feature 1 & 2: Long-term Memory + Named Tool Nodes

Two enhancements to the agent:

1. **Long-term memory** — after each session the agent stores Q&A pairs and retrieved game facts into a ChromaDB-backed vector store. At the start of every new query it recalls semantically similar past knowledge, evaluates its usefulness with an LLM call, and injects the relevant context into the system prompt.

2. **Named tool nodes** — each tool is now an explicit, named node in the state machine graph (`retrieve_game`, `evaluate_retrieval`, `game_web_search`) instead of a single generic `tool_executor`. A `tool_router` step reads the LLM's chosen tool call and routes to the matching node.

New execution graph:
```
__entry__ → memory_recall → message_prep → llm_processor
                                               ↓ (tool call)
                                           tool_router
                                          ↙    ↓    ↘
                               retrieve_game  evaluate_retrieval  game_web_search
                                          ↘    ↓    ↙
                                           llm_processor  (loop)
                                               ↓ (final answer)
                                           memory_store → __termination__
```

In [10]:
from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager

# VectorStoreManager uses an in-memory ChromaDB client — memories persist
# for the lifetime of this kernel session (across multiple agent.invoke calls)
db = VectorStoreManager(openai_api_key=OPENAI_API_KEY)
ltm = LongTermMemory(db=db)

enhanced_agent = Agent(
    model_name=MODEL,
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    long_term_memory=ltm,
    memory_owner="udaplay_user"
)

print("Enhanced agent ready — long-term memory enabled, named tool nodes active.")

Enhanced agent ready — long-term memory enabled, named tool nodes active.


#### Session 1 — Building long-term memory

Run two queries. The execution trace should show **named tool nodes** (`retrieve_game`, `evaluate_retrieval`, etc.) instead of the old generic `tool_executor`. After each query the agent stores the Q&A pair and retrieved game facts into long-term memory.

In [11]:
session1_queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
]

for query in session1_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)
    run = enhanced_agent.invoke(query, session_id="session_1")

    # Show the execution trace — named tool nodes are now visible
    step_names = [s.step_id for s in run.snapshots]
    print(f"\nExecution trace: {' → '.join(step_names)}")

    final_state = run.get_final_state()
    messages = final_state.get("messages", [])
    if messages:
        print(f"\nAnswer:\n{messages[-1].content}")

print(f"\n\nLong-term memory now contains {len(ltm.vector_store._collection.get()['ids'])} fragments.")


Query: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_router
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_router
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: memory_store
[StateMachine] Terminating: __termination__

Execution trace: __entry__ → memory_recall → message_prep → llm_processor → tool_router → retrieve_game → llm_processor → tool_router → evaluate_retrieval → llm_processor → memory_store

Answer:
Pokémon Gold and Silver were released in 1999 for the Game Boy Color. This information comes from the local game database.

Query: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMac

#### Session 2 — Testing long-term memory recall

A brand-new session (`session_2`) starts with **empty short-term memory**. The agent queries long-term memory before processing, evaluates the recalled content for relevance, and injects it into the system prompt if useful. The answer should reflect knowledge from Session 1 without re-running any tool calls.

In [12]:
related_query = "Tell me about Pokémon games for Game Boy Color"

print(f"{'='*60}")
print(f"Query: {related_query}")
print(f"Session: session_2 (fresh — no prior short-term context)")
print('='*60)

run = enhanced_agent.invoke(related_query, session_id="session_2")

final_state = run.get_final_state()

# Show whether long-term memory was recalled and injected
ltm_context = final_state.get("long_term_context")
if ltm_context:
    print("\n[Long-term memory recalled and injected into context]")
    preview = ltm_context[:400] + ("..." if len(ltm_context) > 400 else "")
    print(f"\nMemory injected:\n{preview}")
else:
    print("\n[Long-term memory searched but no relevant context found]")

# Show execution trace
step_names = [s.step_id for s in run.snapshots]
print(f"\nExecution trace: {' → '.join(step_names)}")

messages = final_state.get("messages", [])
if messages:
    print(f"\nAnswer:\n{messages[-1].content}")

print(f"\nTotal tokens this session: {final_state.get('total_tokens', 0)}")

Query: Tell me about Pokémon games for Game Boy Color
Session: session_2 (fresh — no prior short-term context)
[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_router
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_router
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_router
[StateMachine] Executing step: game_web_search
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: memory_store
[StateMachine] Terminating: __termination__

[Long-term memory recalled and injected into context]

Memory injected:
Retrieved game facts: - Platform: Game Boy Color, Name: Pokémon Gold and Silver, Year: 1999, Description: Second-generation Pokémon games introducing new